# BERT Fine-tuning on Colab (Step 5)

This notebook performs BERT fine-tuning using a GPU runtime (Colab / Kaggle). It installs dependencies, loads the dataset, trains `bert-base-uncased`, evaluates, saves the model and tokenizer, and prepares a downloadable archive.

Run cells in order. The training cell will use the GPU and will abort if no CUDA device is available.

In [ ]:
# Install required packages (uncomment to run).
# On Colab these are usually preinstalled but this ensures latest versions.
!pip install -q transformers datasets accelerate 'torch>=2.1' sentencepiece

In [ ]:
# Standard imports
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import joblib

import torch
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import Dataset, DatasetDict
import evaluate


In [ ]:
# Check GPU
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device name:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('No CUDA device detected — switch runtime to GPU before continuing.')

In [ ]:
# Load dataset: try repository path first, else prompt for upload.
local_paths = [
    'dataset/student_emotions_cleaned.csv',
    '/content/student_emotions_cleaned.csv',
]
dataset_path = None
for p in local_paths:
    if os.path.exists(p):
        dataset_path = p
        break

if dataset_path is None:
    from google.colab import files
    print('Please upload the dataset file `student_emotions_cleaned.csv` when prompted.')
    uploaded = files.upload()
    if uploaded:
        dataset_path = next(iter(uploaded.keys()))

print('Loading dataset from:', dataset_path)
df = pd.read_csv(dataset_path)
print('Rows:', len(df))
display(df.head())

In [ ]:
# Prepare labels: ensure a text column and encode target_emotion into integer ids
if 'text' not in df.columns:
    raise SystemExit('Dataset must contain a `text` column.')

if 'target_emotion' not in df.columns:
    raise ValueError('Dataset must contain a `target_emotion` column.')

le = LabelEncoder()
df['target_emotion_id'] = le.fit_transform(df['target_emotion'].astype(str))
label_names = list(le.classes_)
num_labels = len(label_names)
print('Number of labels:', num_labels)
print('Emotion labels:', label_names)

joblib.dump(le, '/content/label_encoder.joblib')
print('Saved LabelEncoder to /content/label_encoder.joblib')


In [ ]:
# Split into train/validation/test if not already split
if {'train', 'validation', 'test'}.issubset(df.columns):
    # assume pre-split with those column names containing booleans or tags
    train_df = df[df['train'] == True] if df['train'].dtype == bool else df[df['split'] == 'train'] if 'split' in df.columns else df.sample(frac=0.8, random_state=42)
    val_df = df[df['validation'] == True] if 'validation' in df.columns else df.sample(frac=0.1, random_state=42)
    test_df = df[df['test'] == True] if 'test' in df.columns else df.sample(frac=0.1, random_state=42)
else:
    # shuffle and split 80/10/10
    df_shuffled = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
    n = len(df_shuffled)
    n_train = int(0.8 * n)
    n_val = int(0.1 * n)
    train_df = df_shuffled.iloc[:n_train].reset_index(drop=True)
    val_df = df_shuffled.iloc[n_train : n_train + n_val].reset_index(drop=True)
    test_df = df_shuffled.iloc[n_train + n_val :].reset_index(drop=True)

print('Train / Val / Test sizes:', len(train_df), len(val_df), len(test_df))

In [ ]:
# Create Hugging Face datasets and tokenize
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True, padding=True, max_length=128)

hf_train = Dataset.from_pandas(train_df[['text','target_emotion_id']])
hf_val = Dataset.from_pandas(val_df[['text','target_emotion_id']])
hf_test = Dataset.from_pandas(test_df[['text','target_emotion_id']])

hf_train = hf_train.map(lambda x: tokenize_batch(x), batched=True)
hf_val = hf_val.map(lambda x: tokenize_batch(x), batched=True)
hf_test = hf_test.map(lambda x: tokenize_batch(x), batched=True)

hf_train = hf_train.rename_column('target_emotion_id', 'labels')
hf_val = hf_val.rename_column('target_emotion_id', 'labels')
hf_test = hf_test.rename_column('target_emotion_id', 'labels')

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

hf_train = hf_train.remove_columns([c for c in hf_train.column_names if c not in tokenizer.model_input_names + ['labels']])
hf_val = hf_val.remove_columns([c for c in hf_val.column_names if c not in tokenizer.model_input_names + ['labels']])
hf_test = hf_test.remove_columns([c for c in hf_test.column_names if c not in tokenizer.model_input_names + ['labels']])

hf_train.set_format(type='torch')
hf_val.set_format(type='torch')
hf_test.set_format(type='torch')

print('Datasets ready. Example tokenized input:')
print(hf_train[0])


In [ ]:
# Build model with correct number of labels and label mapping
id2label = {str(i): label_names[i] for i in range(num_labels)}
label2id = {v: int(k) for k, v in id2label.items()}
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
model.to('cuda')

import evaluate
metric_acc = evaluate.load('accuracy')
metric_f1 = evaluate.load('f1')
metric_precision = evaluate.load('precision')
metric_recall = evaluate.load('recall')


def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    labels = p.label_ids
    acc = metric_acc.compute(predictions=preds, references=labels)['accuracy']
    f1 = metric_f1.compute(predictions=preds, references=labels, average='weighted')['f1']
    prec = metric_precision.compute(predictions=preds, references=labels, average='weighted')['precision']
    rec = metric_recall.compute(predictions=preds, references=labels, average='weighted')['recall']
    return {'accuracy': acc, 'f1': f1, 'precision': prec, 'recall': rec}


In [ ]:
# Training arguments
output_dir = '/content/bert_emotion'
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_strategy='epoch',
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [ ]:
# Train (this will run on GPU).
train_result = trainer.train()
print('Training finished')
print(train_result)

In [ ]:
# Evaluate on test set
predictions = trainer.predict(hf_test)
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)
from sklearn.metrics import classification_report, confusion_matrix
report = classification_report(y_true, y_pred, digits=4)
cm = confusion_matrix(y_true, y_pred)
print('Test classification report:\n')
print(report)
print('Confusion matrix:\n')
print(cm)

In [ ]:
# Save model and tokenizer to output_dir and create zip for download
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

# zip the folder
import shutil
archive_path = '/content/bert_emotion_archive'
shutil.make_archive(archive_path, 'zip', output_dir)
print('Saved and archived to', archive_path + '.zip')

# Provide a download link (Colab)
from google.colab import files
files.download(archive_path + '.zip')